# Liquid Rocket Motor Design Calculator

A first-pass design tool for liquid bipropellant rocket engines.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
from lrd import propellants, nozzle, combustion, injector, tanks, cooling, performance
from lrd.utils import pa_to_psi, psi_to_pa, n_to_lb, m_to_in

## 1. Design Requirements

In [ ]:
# --- Design inputs ---
THRUST_TARGET = 10_000      # N (~2,250 lbf)
CHAMBER_PRESSURE = psi_to_pa(300)  # Pa  (300 psi)
PROPELLANT = "LOX/RP-1"
AMBIENT_PRESSURE = 101325   # Pa (sea level)
BURN_TIME = 30              # seconds

print(f"Target thrust:     {THRUST_TARGET:,.0f} N  ({n_to_lb(THRUST_TARGET):,.0f} lbf)")
print(f"Chamber pressure:  {pa_to_psi(CHAMBER_PRESSURE):.0f} psi  ({CHAMBER_PRESSURE/1e6:.2f} MPa)")
print(f"Propellant:        {PROPELLANT}")
print(f"Burn time:         {BURN_TIME} s")

## 2. Propellant Properties

In [ ]:
prop = propellants.get_propellant(PROPELLANT)

print(f"Available propellants: {propellants.list_propellants()}")
print()
for key, val in prop.items():
    print(f"  {key:>15s}: {val}")

## 3. Nozzle Design

In [ ]:
gamma = prop["gamma"]
c_star = prop["c_star"]
pe_pc = AMBIENT_PRESSURE / CHAMBER_PRESSURE

# Expansion ratio and thrust coefficient
eps = nozzle.expansion_ratio_from_pressure(gamma, pe_pc)
cf = nozzle.thrust_coefficient(gamma, eps, pe_pc, pe_pc)
isp = performance.specific_impulse(c_star, cf)
Me = nozzle.exit_mach(gamma, eps)

# Throat sizing
At = THRUST_TARGET / (cf * CHAMBER_PRESSURE)
dt = nozzle.diameter_from_area(At)
Ae = nozzle.exit_area(At, eps)
de = nozzle.diameter_from_area(Ae)

print(f"Expansion ratio:     {eps:.2f}")
print(f"Exit Mach number:    {Me:.2f}")
print(f"Thrust coefficient:  {cf:.3f}")
print(f"Specific impulse:    {isp:.1f} s")
print(f"Throat diameter:     {m_to_in(dt):.2f} in  ({dt*1000:.1f} mm)")
print(f"Exit diameter:       {m_to_in(de):.2f} in  ({de*1000:.1f} mm)")

In [ ]:
# Nozzle contour plot
x, y = nozzle.nozzle_contour(dt / 2, eps, half_angle_deg=15)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x * 1000, y * 1000, "b-", linewidth=2)
ax.plot(x * 1000, -y * 1000, "b-", linewidth=2)
ax.set_xlabel("Axial distance from throat [mm]")
ax.set_ylabel("Radius [mm]")
ax.set_title("Nozzle Contour (conical, 15° half-angle)")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Combustion Chamber Sizing

In [ ]:
CONTRACTION_RATIO = 3.0

l_star = combustion.l_star_typical(PROPELLANT)
Vc = combustion.chamber_volume(At, l_star)
chamber = combustion.chamber_dimensions(Vc, CONTRACTION_RATIO, At)
ts = combustion.stay_time(l_star, c_star, CHAMBER_PRESSURE, prop["T_c"], prop["mol_weight"])

print(f"L* (characteristic length): {l_star:.2f} m")
print(f"Chamber volume:    {Vc*1e6:.1f} cm³")
print(f"Chamber diameter:  {m_to_in(chamber['diameter']):.2f} in  ({chamber['diameter']*1000:.1f} mm)")
print(f"Chamber length:    {m_to_in(chamber['length']):.2f} in  ({chamber['length']*1000:.1f} mm)")
print(f"Residence time:    {ts*1000:.2f} ms")

## 5. Injector Sizing

In [ ]:
# Mass flow rates
mdot = performance.mass_flow_rate(THRUST_TARGET, isp)
flows = performance.mixture_ratio_flows(mdot, prop["of_ratio"])

print(f"Total mdot:  {mdot:.3f} kg/s")
print(f"Ox mdot:     {flows['mdot_ox']:.3f} kg/s")
print(f"Fuel mdot:   {flows['mdot_fuel']:.3f} kg/s")

In [ ]:
CD = 0.7
INJ_DP_FRACTION = 0.20  # 20% of Pc
dp_inj = INJ_DP_FRACTION * CHAMBER_PRESSURE

# Ox orifices (2 mm diameter)
d_ox = 0.002  # m
n_ox = injector.orifice_count(flows["mdot_ox"], CD, d_ox, dp_inj, prop["ox_density"])

# Fuel orifices (1.5 mm diameter)
d_fuel = 0.0015  # m
n_fuel = injector.orifice_count(flows["mdot_fuel"], CD, d_fuel, dp_inj, prop["fuel_density"])

check = injector.check_pressure_drop_ratio(dp_inj, CHAMBER_PRESSURE)

print(f"Injector ΔP:       {pa_to_psi(dp_inj):.0f} psi  ({dp_inj/1e6:.2f} MPa)")
print(f"Ox orifices:       {n_ox} × {d_ox*1000:.1f} mm dia")
print(f"Fuel orifices:     {n_fuel} × {d_fuel*1000:.1f} mm dia")
print(f"ΔP check:          {check['message']}")

## 6. Feed System & Tanks

In [ ]:
m_ox = flows["mdot_ox"] * BURN_TIME
m_fuel = flows["mdot_fuel"] * BURN_TIME

v_ox = tanks.tank_volume(m_ox, prop["ox_density"])
v_fuel = tanks.tank_volume(m_fuel, prop["fuel_density"])

feed_p = tanks.feed_pressure(CHAMBER_PRESSURE, dp_inj)

# Tank mass estimates (Al 6061-T6, yield ~276 MPa)
AL_YIELD = 276e6  # Pa
ox_tank = tanks.tank_mass_thin_wall(v_ox, feed_p, AL_YIELD)
fuel_tank = tanks.tank_mass_thin_wall(v_fuel, feed_p, AL_YIELD)

print(f"Oxidizer mass:     {m_ox:.2f} kg")
print(f"Fuel mass:         {m_fuel:.2f} kg")
print(f"Total propellant:  {m_ox + m_fuel:.2f} kg")
print()
print(f"Ox tank volume:    {v_ox*1000:.2f} L")
print(f"Fuel tank volume:  {v_fuel*1000:.2f} L")
print(f"Feed pressure:     {pa_to_psi(feed_p):.0f} psi  ({feed_p/1e6:.2f} MPa)")
print()
print(f"Ox tank mass:      {ox_tank['mass']:.2f} kg  (wall: {ox_tank['wall_thickness']*1000:.2f} mm)")
print(f"Fuel tank mass:    {fuel_tank['mass']:.2f} kg  (wall: {fuel_tank['wall_thickness']*1000:.2f} mm)")

## 7. Cooling Estimates

In [ ]:
# Bartz heat flux at throat
MU_GAS = 8e-5       # Pa·s (typical for combustion products)
PR_GAS = 0.5        # Prandtl number

heat = cooling.bartz_heat_flux(
    CHAMBER_PRESSURE, c_star, dt, prop["T_c"], gamma, MU_GAS, PR_GAS
)

print(f"Throat heat transfer coeff:  {heat['h_g']:.0f} W/(m²·K)")
print(f"Throat heat flux:            {heat['heat_flux']/1e6:.2f} MW/m²")

# Film cooling estimate (using RP-1 as coolant, cp ≈ 2000 J/(kg·K), ΔT ≈ 200 K)
film_flux = cooling.film_cooling_fraction(heat["heat_flux"], 2000, 200)
print(f"Film cooling mass flux:      {film_flux:.2f} kg/(m²·s)")

## 8. Performance Summary

In [ ]:
summary = performance.engine_summary(
    thrust_target=THRUST_TARGET,
    chamber_pressure=CHAMBER_PRESSURE,
    propellant_key=PROPELLANT,
    ambient_pressure=AMBIENT_PRESSURE,
    burn_time=BURN_TIME,
)

print("=" * 50)
print("    ENGINE DESIGN SUMMARY")
print("=" * 50)
print(f"  Propellant:          {summary['propellant']}")
print(f"  Thrust:              {summary['thrust']:,.0f} N  ({n_to_lb(summary['thrust']):,.0f} lbf)")
print(f"  Isp:                 {summary['isp']:.1f} s")
print(f"  Chamber pressure:    {pa_to_psi(summary['chamber_pressure']):.0f} psi")
print(f"  Expansion ratio:     {summary['expansion_ratio']:.2f}")
print(f"  C_F:                 {summary['thrust_coefficient']:.3f}")
print(f"  c*:                  {summary['c_star']:.0f} m/s")
print("-" * 50)
print(f"  Throat diameter:     {m_to_in(summary['throat_diameter']):.2f} in")
print(f"  Exit diameter:       {m_to_in(summary['exit_diameter']):.2f} in")
print(f"  Chamber diameter:    {m_to_in(summary['chamber_diameter']):.2f} in")
print(f"  Chamber length:      {m_to_in(summary['chamber_length']):.2f} in")
print("-" * 50)
print(f"  Mass flow (total):   {summary['mdot_total']:.3f} kg/s")
print(f"  Mass flow (ox):      {summary['mdot_ox']:.3f} kg/s")
print(f"  Mass flow (fuel):    {summary['mdot_fuel']:.3f} kg/s")
print(f"  O/F ratio:           {summary['of_ratio']:.2f}")
print("-" * 50)
print(f"  Burn time:           {summary['burn_time']} s")
print(f"  Ox mass:             {summary['ox_mass']:.2f} kg")
print(f"  Fuel mass:           {summary['fuel_mass']:.2f} kg")
print(f"  Feed pressure:       {pa_to_psi(summary['feed_pressure']):.0f} psi")
print("=" * 50)